<a href="https://colab.research.google.com/github/ArifahZhafirah/PEMBELAJARAN-MESIN/blob/main/JS03/TUGAS/PEMMES%20TUGAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Pisahkan variabel yang dapat digunakan dan tidak dapat digunakan

In [4]:
import pandas as pd

df = pd.read_csv("wbc.csv", sep=";")

# "Unnamed: 32" kosong (akibat delimiter di akhir baris csv) -> tidak dapat digunakan
df = df.drop(columns=["Unnamed: 32"])

# "id" hanya identifier, tidak relevan sebagai fitur -> tidak dapat digunakan
X = df.drop(columns=["id", "diagnosis"])   # variabel yang dapat digunakan (30 fitur numerik)
y_raw = df["diagnosis"]                    # target

print("Jumlah fitur yang dapat digunakan:", X.shape[1])
X.info()

Jumlah fitur yang dapat digunakan: 30
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   radius_mean              569 non-null    float64
 1   texture_mean             569 non-null    float64
 2   perimeter_mean           569 non-null    float64
 3   area_mean                569 non-null    float64
 4   smoothness_mean          569 non-null    float64
 5   compactness_mean         569 non-null    float64
 6   concavity_mean           569 non-null    float64
 7   concave points_mean      569 non-null    float64
 8   symmetry_mean            569 non-null    float64
 9   fractal_dimension_mean   569 non-null    float64
 10  radius_se                569 non-null    float64
 11  texture_se               569 non-null    float64
 12  perimeter_se             569 non-null    float64
 13  area_se                  569 non-null    f

#2. Encoding kolom "diagnosis"

In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y_raw)   # B -> 0, M -> 1

print(dict(zip(le.classes_, le.transform(le.classes_))))
print(pd.Series(y_raw).value_counts())

{'B': np.int64(0), 'M': np.int64(1)}
diagnosis
B    357
M    212
Name: count, dtype: int64


#3. Standardisasi kolom numerik

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# pisah train/test dulu, sebelum scaling dan seleksi fitur
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
# scaler akan dipanggil lewat pipeline pada langkah 6, bukan manual di sini

#4. Seleksi fitur dengan SelectKBest

In [7]:
from sklearn.feature_selection import SelectKBest, f_classif

# SelectKBest juga akan dijalankan lewat pipeline (langkah 6)
# di sini contoh melihat skor F setiap fitur pada data latih yang sudah discaling
X_train_scaled = scaler.fit_transform(X_train)
selector = SelectKBest(score_func=f_classif, k=10)
selector.fit(X_train_scaled, y_train)

skor = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
print(skor.head(10))

concave points_worst    733.724933
perimeter_worst         717.246487
radius_worst            692.861395
concave points_mean     684.526845
perimeter_mean          548.413236
area_worst              522.188947
radius_mean             511.274848
area_mean               444.857518
concavity_mean          397.592082
concavity_worst         319.507776
dtype: float64


#5. Pengujian dengan model Logistic Regression

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(selector.transform(X_train_scaled), y_train)

X_test_scaled = scaler.transform(X_test)
pred = model.predict(selector.transform(X_test_scaled))

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.956140350877193
              precision    recall  f1-score   support

           0       0.95      0.99      0.97        72
           1       0.97      0.90      0.94        42

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



#6. Menggabungkan semua proses dengan Pipeline

In [9]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("sel", SelectKBest(score_func=f_classif, k=10)),
    ("clf", LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.956140350877193
              precision    recall  f1-score   support

           0       0.95      0.99      0.97        72
           1       0.97      0.90      0.94        42

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



#7. Menentukan jumlah fitur terbaik (k)

In [12]:
from sklearn.model_selection import cross_val_score

# Cari k terbaik dengan rentang lebih rinci
hasil = []
for k in range(5, 31):
    pipe_k = Pipeline([
        ("scaler", StandardScaler()),
        ("sel", SelectKBest(score_func=f_classif, k=k)),
        ("clf", LogisticRegression(max_iter=1000))
    ])
    skor = cross_val_score(pipe_k, X_train, y_train, cv=5)
    hasil.append((k, skor.mean(), skor.std()))

hasil_df = pd.DataFrame(hasil, columns=["k", "mean_accuracy", "std_accuracy"])

# Pilih k dengan akurasi tertinggi (dan paling stabil jika ada yang seri)
best_k = int(hasil_df.sort_values(
    ["mean_accuracy", "std_accuracy"], ascending=[False, True]
).iloc[0]["k"])
print("Jumlah fitur terbaik (k):", best_k)

# Latih ulang pipeline dengan k terbaik
pipe_best = Pipeline([
    ("scaler", StandardScaler()),
    ("sel", SelectKBest(score_func=f_classif, k=best_k)),
    ("clf", LogisticRegression(max_iter=1000))
])
pipe_best.fit(X_train, y_train)
pred = pipe_best.predict(X_test)
print("Accuracy pada data test:", accuracy_score(y_test, pred))

# Tampilkan nama-nama fitur yang terpilih
mask = pipe_best.named_steps["sel"].get_support()
fitur_terpilih = sorted(
    zip(X.columns[mask], pipe_best.named_steps["sel"].scores_[mask]),
    key=lambda x: x[1], reverse=True
)
print(f"\n{len(fitur_terpilih)} fitur terbaik:")
for nama, skor in fitur_terpilih:
    print(f"{nama:30s} {skor:.2f}")

Jumlah fitur terbaik (k): 19
Accuracy pada data test: 0.9824561403508771

19 fitur terbaik:
concave points_worst           733.72
perimeter_worst                717.25
radius_worst                   692.86
concave points_mean            684.53
perimeter_mean                 548.41
area_worst                     522.19
radius_mean                    511.27
area_mean                      444.86
concavity_mean                 397.59
concavity_worst                319.51
compactness_mean               263.56
compactness_worst              238.20
radius_se                      205.43
perimeter_se                   193.17
area_se                        180.56
texture_worst                  126.12
smoothness_worst               103.72
symmetry_worst                 100.56
texture_mean                   93.48
